In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

os.environ["CUDA_LAUNCH_BLOCKING"] = "0" 
os.environ["GRB_LICENSE_FILE"] = "/usr0/home/naveenr/gurobi.lic" 
os.environ['MKL_THREADING_LAYER'] = "GNU"
import torch 

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [3]:
from concept_abstraction.training import *
from concept_abstraction.selection import *
from concept_abstraction.concept_bank import *
from concept_abstraction.env_utils import *
from concept_abstraction.environments import *
from concept_abstraction.utils import *
import sys 
import argparse
import secrets
import numpy as np 
import random 
import os
from stable_baselines3 import PPO
import pickle
import resource
import time 

In [4]:
is_jupyter = 'ipykernel' in sys.modules
is_main = __name__ == "__main__"

In [485]:
if is_main:
    seed = 43
    environment_string = "boxing"
    gold_timesteps = 30_000_000
    training_timesteps = 250_000 
    num_concepts_selected = 48
    out_folder = "basic"
    method = "lp" 


In [486]:
if is_main:
    concept_list, processed_concepts = get_concepts(environment_string,"human_selected_binary",seed)
    num_concepts_selected = min(num_concepts_selected,len(concept_list))
    ground_truth_env, ground_truth_gym_env = get_environment(environment_string, None, seed)   
    model_name = "../../results/models/env={}_training={}_seed={}.zip".format(environment_string,gold_timesteps,seed)
    if os.path.exists(model_name):
        groundtruth_model = PPO.load(model_name)
    model_name = "../../results/q_estimates/env={}_training={}_seed={}_selection={}_source={}.pkl".format(environment_string,gold_timesteps,seed,"q_value","human_selected_binary")
    if os.path.exists(model_name):
        q_estimates = pickle.load(open(model_name,"rb"))

In [492]:
q_estimates = q_estimate_list

In [493]:
subset_concept, idx = lp_based_selection(ground_truth_gym_env,concept_list,num_concepts_selected,"q_value",q_estimates,"human_selected_binary")

18
0
100
200
300
400
500
600
700
800
900
1000
1100


In [481]:
subset_concept, idx = policy_coverage_selection_lp(
    ground_truth_gym_env,
    concept_list,
    num_concepts_selected,
    groundtruth_model)

There are 8000 observations
Coverage 0.9851


In [482]:
len(idx)

48

In [483]:
subset_concept, idx = policy_coverage_selection_lp_advantage(
    ground_truth_gym_env,
    concept_list,
    num_concepts_selected,
    groundtruth_model)

Starting stable training for sparse rewards...
Total of 200000 steps
Step 0/200000, Loss mean: 0.0000
Step 5000/200000, Loss mean: 0.8814
Step 10000/200000, Loss mean: 0.7551
Step 15000/200000, Loss mean: 0.8678
Step 20000/200000, Loss mean: 0.5986
Updated target network at step 500, Avg recent loss: 0.5936
Step 25000/200000, Loss mean: 0.2069
Step 30000/200000, Loss mean: 0.1720
Step 35000/200000, Loss mean: 0.1701
Step 40000/200000, Loss mean: 0.1706
Updated target network at step 1000, Avg recent loss: 0.1706
Step 45000/200000, Loss mean: 0.0824
Step 50000/200000, Loss mean: 0.0754
Step 55000/200000, Loss mean: 0.0768
Step 60000/200000, Loss mean: 0.0919
Updated target network at step 1500, Avg recent loss: 0.0920
Step 65000/200000, Loss mean: 0.0813
Step 70000/200000, Loss mean: 0.0909
Episode 25, Avg Reward: 70.12, Loss (mean/std/max): 0.09/0.05/0.21, Epsilon: 0.041
Step 75000/200000, Loss mean: 0.0841
Step 80000/200000, Loss mean: 0.0887
Updated target network at step 2000, Avg r

In [484]:
len(idx)

48